# Jupiter xStock Paper Trading — Learn the APIs

**Purpose**: Learn Jupiter DEX APIs for the Yagnum ERR research pipeline (Slice 3)
**Cost**: $0 — all read-only, no wallet or SOL needed
**Time**: ~15 min to run all cells

| Step | What You Learn | API Used |
|------|---------------|----------|
| 0 | Install deps | pip |
| 1 | Set up Jupiter API key | Config |
| 2 | Discover xStock tokens | Token Search |
| 3 | Fetch live prices | Price API v3 |
| 4 | Get swap quotes | Quote API |
| 5 | Paper trade simulation | Quote + Portfolio |
| 6 | ERR gap concept | Conceptual |


## Step 0 — Install Dependencies

Run once. Everything except `httpx` may already be installed.


In [1]:
# One-time install
!pip install httpx python-dotenv --quiet

import httpx
import json
import os
import time
from datetime import datetime, timezone
from dotenv import load_dotenv

print("All imports OK")
print("httpx version:", httpx.__version__)


All imports OK
httpx version: 0.28.1



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Jupiter API Key

Get a **free** key at [developers.jup.ag/portal](https://developers.jup.ag/portal):
1. Sign up
2. Create a Team
3. Generate API Key (starts with `jup_...`)
4. Paste below

Without a key you get 0.5 RPS (1 call every 2s). With a free key: 1 RPS.


In [2]:
# Paste your key here, or load from .env
JUP_API_KEY = ""

load_dotenv("../.env")
if not JUP_API_KEY:
    JUP_API_KEY = os.getenv("JUP_API_KEY", "")

# Endpoints
PRICE_URL  = "https://api.jup.ag/price/v3"
QUOTE_URL  = "https://api.jup.ag/swap/v1/quote"
SEARCH_URL = "https://api.jup.ag/tokens/v2/search"
USDC_MINT  = "EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v"
SOL_MINT   = "So11111111111111111111111111111111111111112"

def hdrs():
    h = {"Accept": "application/json"}
    if JUP_API_KEY:
        h["x-api-key"] = JUP_API_KEY
    return h

DELAY = 1.2 if JUP_API_KEY else 2.2

if JUP_API_KEY:
    print("API Key: SET (" + JUP_API_KEY[:8] + "...)")
    print("Rate limit: 1 RPS (free tier)")
else:
    print("No API key - using keyless access (0.5 RPS)")
    print("Get a free key at: https://developers.jup.ag/portal")

print()
print("Endpoints:")
print("  Price: ", PRICE_URL)
print("  Quote: ", QUOTE_URL)
print("  Search:", SEARCH_URL)


API Key: SET (jup_7db9...)
Rate limit: 1 RPS (free tier)

Endpoints:
  Price:  https://api.jup.ag/price/v3
  Quote:  https://api.jup.ag/swap/v1/quote
  Search: https://api.jup.ag/tokens/v2/search


## Step 2 — Discover xStock Tokens

xStocks = tokenized US equities on Solana (issued by Backed Finance).
Each has a unique **mint address** (like a contract address).


In [3]:
# Search for real xStock tokens
# The true Backed Finance symbols end in 'x' (e.g. AAPLx, NVDAx)
symbols = ["AAPLx", "TSLAx", "NVDAx", "MSFTx", "AMZNx"]
found = {}

header = "{:<10} {:<25} {:<10} {}".format("Symbol", "Name", "Decimals", "Mint Address")
print(header)
print("-" * 85)

with httpx.Client() as c:
    for sym in symbols:
        time.sleep(DELAY)
        try:
            r = c.get(SEARCH_URL, params={"query": sym}, headers=hdrs(), timeout=10)
            if r.status_code == 200:
                data = r.json()
                tokens = data if isinstance(data, list) else data.get("tokens", [])
                matched = False
                for t in tokens:
                    # Filter for verified xStocks to avoid meme coins
                    tags = t.get("tags", [])
                    if t.get("symbol", "").lower() == sym.lower() and ("xstocks" in tags or "stocks" in tags or "rwa" in tags):
                        mint = t.get("id", t.get("address", "?"))
                        name = t.get("name", "?")
                        dec = t.get("decimals", "?")
                        found[sym] = {"mint": mint, "name": name, "decimals": dec}
                        row = "{:<10} {:<25} {:<10} {}".format(sym, name, str(dec), mint)
                        print(row)
                        matched = True
                        break
                if not matched:
                    print("{:<10} (not found or no verified tags)".format(sym))
            elif r.status_code == 429:
                print("{:<10} (rate limited - wait and rerun)".format(sym))
            else:
                print("{:<10} HTTP {}".format(sym, r.status_code))
        except Exception as e:
            print("{:<10} Error: {}".format(sym, e))

print()
print("Found {} real xStock tokens".format(len(found)))


Symbol     Name                      Decimals   Mint Address
-------------------------------------------------------------------------------------
AAPLx      Apple xStock              8          XsbEhLAtcf6HdfpFZ5xEMdqW8nfAvcsP5bdudRLJzJp
TSLAx      Tesla xStock              8          XsDoVfqeBukxuZHWhdvWHBhgEHjGNst4MLodqsJHzoB
NVDAx      NVIDIA xStock             8          Xsc9qvGR1efVDFGLrVsmkzv3qi45LTBjeUKSPmx9qEh
MSFTx      Microsoft xStock          8          XspzcW1PRtgf6Wj92HCiZdjzKCyFekVD8P5Ueh3dRMX
AMZNx      Amazon xStock             8          Xs3eBt7uRfJX8QUs4suhyU8p2M6DoUDrJyWBa8LLZsg

Found 5 real xStock tokens


## Step 3 — Fetch Live Prices

Jupiter Price API v3 returns USD prices for any Solana token, 24/7.

```
GET https://api.jup.ag/price/v3?ids=mint1,mint2
```

This gives us **P_JUP** — the price the ERR locks against.


In [4]:
# Get live prices
mints_to_price = {SOL_MINT: "SOL"}
for sym, info in found.items():
    if info["mint"] != "?":
        mints_to_price[info["mint"]] = sym

with httpx.Client() as c:
    time.sleep(DELAY)
    r = c.get(PRICE_URL,
              params={"ids": ",".join(mints_to_price.keys())},
              headers=hdrs(), timeout=10)

    print("HTTP", r.status_code)
    print()

    if r.status_code == 200:
        data = r.json()
        if "data" in data:
            data = data["data"]
            
        print("{:<10} {:>16}".format("Token", "Price (USD)"))
        print("-" * 30)
        for mint, info in data.items():
            sym = mints_to_price.get(mint, mint[:8])
            price = info.get("price") or info.get("usdPrice") if isinstance(info, dict) else None
            if price:
                print("{:<10} ${:>14,.6f}".format(sym, float(price)))
            else:
                print("{:<10} (no price found)".format(sym))
    elif r.status_code == 429:
        print("Rate limited. Wait a few seconds and rerun.")
    else:
        print(r.text[:300])


HTTP 200

Token           Price (USD)
------------------------------
SOL        $     76.045470
AMZNx      $    266.387727
TSLAx      $    340.326547
AAPLx      $    304.943287
NVDAx      $    225.046021
MSFTx      $    494.893605


## Step 4 — Get a Swap Quote (Read-Only)

The Quote API shows what a swap **would** produce without executing it.

```
GET https://api.jup.ag/swap/v1/quote
    ?inputMint=SOL_MINT
    &outputMint=USDC_MINT
    &amount=1000000000      (1 SOL = 10^9 lamports)
    &slippageBps=50         (0.5% max slippage)
```

**Nothing is executed.** This is a read-only price check.


In [5]:
# Quote: 1 SOL -> USDC
with httpx.Client() as c:
    time.sleep(DELAY)
    r = c.get(QUOTE_URL, params={
        "inputMint": SOL_MINT,
        "outputMint": USDC_MINT,
        "amount": "1000000000",
        "slippageBps": 50,
    }, headers=hdrs(), timeout=15)

    if r.status_code == 200:
        q = r.json()
        out_raw = int(q.get("outAmount", 0))
        out_usd = out_raw / 1_000_000  # USDC = 6 decimals

        print("QUOTE: 1 SOL -> USDC")
        print("  Out:          {:,} raw = ${:,.6f}".format(out_raw, out_usd))
        print("  Price Impact:", q.get("priceImpactPct", "N/A"))
        print("  Slippage:    ", q.get("slippageBps", "?"), "bps")
        for i, hop in enumerate(q.get("routePlan", [])):
            si = hop.get("swapInfo", {})
            label = si.get("label", "?")
            pct = hop.get("percent", "?")
            print("  Route Hop {}:  {} ({}%)".format(i + 1, label, pct))

        print()
        print("P_JUP = ${:,.6f}  <-- ERR locks this price".format(out_usd))
    else:
        print("HTTP {}: {}".format(r.status_code, r.text[:200]))


QUOTE: 1 SOL -> USDC
  Out:          76,073,368 raw = $76.073368
  Price Impact: 0
  Slippage:     50 bps
  Route Hop 1:  ZeroFi (100%)

P_JUP = $76.073368  <-- ERR locks this price


## Step 4b — Quote an xStock (if found)

This is the real Yagnum scenario: *"Alice sells 1 xAAPL on Saturday"*


In [6]:
# Quote first discovered xStock -> USDC
if found:
    sym = list(found.keys())[0]
    info = found[sym]
    dec = int(info["decimals"]) if info["decimals"] != "?" else 8
    amt = 10 ** dec  # 1 token

    print("QUOTE: 1 {} -> USDC".format(sym))
    print("  Mint:    ", info["mint"])
    print("  Decimals:", dec)
    print("  Amount:  ", "{:,}".format(amt), "raw")
    print()

    with httpx.Client() as c:
        time.sleep(DELAY)
        r = c.get(QUOTE_URL, params={
            "inputMint": info["mint"],
            "outputMint": USDC_MINT,
            "amount": str(amt),
            "slippageBps": 100,
        }, headers=hdrs(), timeout=15)

        if r.status_code == 200:
            q = r.json()
            out_usd = int(q.get("outAmount", 0)) / 1_000_000
            print("  Result: ${:,.6f}".format(out_usd))
            print("  Impact:", q.get("priceImpactPct", "N/A"))
            for i, hop in enumerate(q.get("routePlan", [])):
                si = hop.get("swapInfo", {})
                print("  Hop {}:   {} ({}%)".format(i+1, si.get("label", "?"), hop.get("percent", "?")))
            print()
            print("P_JUP for 1 {} = ${:,.6f}".format(sym, out_usd))
        else:
            print("HTTP {}: {}".format(r.status_code, r.text[:200]))
else:
    print("No xStocks found. Rerun Step 2.")


QUOTE: 1 AAPLx -> USDC
  Mint:     XsbEhLAtcf6HdfpFZ5xEMdqW8nfAvcsP5bdudRLJzJp
  Decimals: 8
  Amount:   100,000,000 raw

  Result: $304.634961
  Impact: 0.0005345871877754334064340268
  Hop 1:   Byreal (100%)

P_JUP for 1 AAPLx = $304.634961


## Step 5 — Paper Trade Simulation

Buy tokens at live price, wait, check new price, calculate P&L.
**No wallet. No SOL. Pure simulation.**


In [7]:
# Paper trade: BUY 10 SOL, wait, check P&L
portfolio = {"cash": 100_000.0, "positions": []}

with httpx.Client() as c:
    # Entry price
    time.sleep(DELAY)
    r1 = c.get(QUOTE_URL, params={
        "inputMint": SOL_MINT, "outputMint": USDC_MINT,
        "amount": "1000000000", "slippageBps": 50,
    }, headers=hdrs(), timeout=15)

    if r1.status_code != 200:
        print("Error: HTTP", r1.status_code)
    else:
        entry = int(r1.json().get("outAmount", 0)) / 1_000_000
        qty = 10
        cost = qty * entry
        portfolio["cash"] -= cost
        portfolio["positions"].append({
            "sym": "SOL", "qty": qty, "entry": entry,
            "time": datetime.now(timezone.utc).isoformat()
        })

        print("BUY {} SOL @ ${:,.6f}  (cost: ${:,.2f})".format(qty, entry, cost))
        print("Cash remaining: ${:,.2f}".format(portfolio["cash"]))
        print()
        print("Waiting 8s for price movement...")
        time.sleep(8)

        # Exit price
        time.sleep(DELAY)
        r2 = c.get(QUOTE_URL, params={
            "inputMint": SOL_MINT, "outputMint": USDC_MINT,
            "amount": "1000000000", "slippageBps": 50,
        }, headers=hdrs(), timeout=15)

        if r2.status_code == 200:
            curr = int(r2.json().get("outAmount", 0)) / 1_000_000
            delta = curr - entry
            pnl = delta * qty
            pct = delta / entry * 100

            print()
            print("Entry:   ${:,.6f}".format(entry))
            print("Current: ${:,.6f}".format(curr))
            print("Delta:   ${:+,.6f}  ({:+.4f}%)".format(delta, pct))
            print("P&L:     ${:+,.2f}".format(pnl))
            print()
            print("This delta is the ERR gap risk.")
            print("Over a weekend it could be much larger.")


BUY 10 SOL @ $76.080556  (cost: $760.81)
Cash remaining: $99,239.19

Waiting 8s for price movement...

Entry:   $76.080556
Current: $76.082792
Delta:   $+0.002236  (+0.0029%)
P&L:     $+0.02

This delta is the ERR gap risk.
Over a weekend it could be much larger.


## Step 6 — How This Connects to Yagnum ERR

```
Friday 4pm ET:  Trader sells 10 xAAPL on Jupiter
                P_JUP = $187.42  (locked by ERR)
                     |
    [NYSE CLOSED]    |   ERR absorbs this uncertainty
                     |
Monday 9:30am:  Yagnum sells 10 AAPL via Alpaca
                P_MKT = $189.15  (actual fill)

ERR Delta = |189.15 - 187.42| = $1.73 per share
Total ERR = $1.73 x 10 = $17.30 surplus -> refunded to trader
```

### What you now know:

- **Jupiter Quote API** = P_JUP (provisional price, works 24/7)
- **Jupiter Price API** = real-time token prices
- **Alpaca API** (your existing setup) = P_MKT (broker fill, NYSE hours)
- **ERR** = the difference between the two

### Next: build the data collection pipeline
- Log P_JUP values to PostgreSQL over time
- Collect Friday-close vs Monday-open gaps
- Connect Alpaca for the TradFi hedge leg


In [ ]:
# Quick ref: add your Jupiter API key to .env
print("Add to /home/ubuntu/ilabs/jit-ledger/.env:")
print()
print("  JUP_API_KEY=jup_your_key_here")
print()
print("Then restart kernel and rerun Step 1.")
print()
print("Tiers:")
print("  Keyless    0.5 RPS  (no signup)")
print("  Free       1.0 RPS  (free at developers.jup.ag/portal)")
print("  Developer  10  RPS  ($25/mo)")


Unfortunately, no. You cannot use Solana Devnet with Jupiter.

Jupiter's entire purpose is to aggregate liquidity from DEXes (like Raydium, Orca, Meteora, etc.). Since real liquidity and real tokens (like the xStocks or USDC) don't exist on Devnet, Jupiter's API only supports Mainnet.

But don't worry! In Solana, there is a built-in feature called Transaction Simulation that solves exactly this problem.

Here is how you safely practice Buy/Sell execution on Mainnet without spending a dime:

Build the Trade: You ask Jupiter to generate a real Mainnet transaction for you.
Sign It: You sign it with your real wallet.
Simulate It (The Magic Step): Instead of broadcasting it to the network to be mined, you send it to a Solana RPC node and ask it to simulateTransaction.
The network will run your trade against the live blockchain state and return exactly what would have happened (e.g., "Success! You would have paid 0.000005 SOL in gas and received 1 xNVDA").

Nothing is written to the blockchain, and 0 SOL leaves your wallet. This is how algorithmic trading bots safely test their logic before going live.

If you'd like, I can write a Python script that takes you through this exact flow: getting a real quote, building the transaction, and running a safe Mainnet simulation. All you'd need is to generate an empty Solana wallet for testing. Want to proceed with that?